# PRUEBAS

In [0]:
%sql
--USE CATALOG sbsrisk_dev;
--USE SCHEMA bronze;

--CREATE TABLE IF NOT EXISTS bronze_sbs_indice_2024 (
--  _c0 STRING, _c1 STRING, _c2 STRING, _c3 STRING, _c4 STRING, _c5 STRING, _c6 STRING,
--  source_file STRING,
--  periodo_mes STRING
--)
--USING DELTA
--LOCATION 'abfss://bronze@adlsbsriskdev.dfs.core.windows.net/sbsrisk_dev/bronze/bronze_sbs_indice_2024';

In [0]:
"""
import re
from pyspark.sql.functions import lit

# Ajusta esta base al path REAL donde dejaste los excels
base_2024 = "abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/"

# Lista archivos (si tu carpeta es distinta, acá lo verás)
files = [f.path for f in dbutils.fs.ls(base_2024) if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]
print("Archivos encontrados:", len(files))
for f in files[:5]:
    print(f)

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # extrae mes: en2024, fe2024, etc.
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    df = (
        spark.read.format("com.crealytics.spark.excel")
        .option("header", "false")
        .option("inferSchema", "false")
        .option("treatEmptyValuesAsNulls", "true")
        .option("sheetName", "INDICE")
        .load(file_path)
    )

    df = (
        df.withColumn("source_file", lit(file_name))
          .withColumn("periodo_mes", lit(periodo))
    )

    all_dfs.append(df)

# unir todo
df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d)

display(df_final.limit(20))
"""

In [0]:
"""
(
  df_final.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("sbsrisk_dev.bronze.bronze_sbs_indice_2024")
)
"""

In [0]:
"""
%sql
SELECT COUNT(*) AS total_rows
FROM sbsrisk_dev.bronze.bronze_sbs_indice_2024;
"""

In [0]:
# probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/")

In [0]:
#probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/")

In [0]:
# probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/")

In [0]:
"""
import re
from pyspark.sql import functions as F
from pyspark.sql.functions import lit

def read_excel_sheet(path: str, sheet: str):
    ext = path.split(".")[-1].lower()  # xls o xlsx
    wb_type = "xls" if ext == "xls" else "xlsx"

    return (spark.read.format("com.crealytics.spark.excel")
            .option("workbookType", wb_type)
            .option("header", "false")
            .option("inferSchema", "false")
            .option("treatEmptyValuesAsNulls", "true")
            .option("maxColumns", "80")          # clave: índice usa columnas lejanas
            .option("maxRowsInMemory", "20000")
            .option("sheetName", sheet)
            .load(path))

def get_sheet_candidates_from_indice(path: str):
    # Intentamos leer la hoja índice (solo para obtener nombres de hojas)
    indice_names = ["Índice", "INDICE", "ÍNDICE", "Indice", "INDICE " , "ÍNDICE "]
    df_ind = None
    last_err = None

    for sh in indice_names:
        try:
            df_ind = read_excel_sheet(path, sh)
            break
        except Exception as e:
            last_err = e

    if df_ind is None:
        raise Exception(f"No pude leer hoja Índice en {path}. Último error: {last_err}")

    # En Índice: normalmente el nombre de la hoja está en la primera columna (_c0)
    # Sacamos lista única de textos no nulos
    sheet_list = (df_ind
        .select(F.col("_c0").cast("string").alias("sheet_txt"))
        .where(F.col("sheet_txt").isNotNull())
        .withColumn("sheet_txt", F.trim(F.col("sheet_txt")))
        .where(F.length("sheet_txt") > 0)
        .dropDuplicates()
    )

    return sheet_list

# Define aquí las hojas que SÍ nos interesan para el proyecto (puedes ajustar)
TARGET_KEYWORDS = [
    "Ratios de Morosidad",               # tu foco inicial
    "Balance General",
    "Estado de Ganancias",
]

def pick_target_sheets(sheet_list_df):
    cond = None
    for kw in TARGET_KEYWORDS:
        expr = F.lower(F.col("sheet_txt")).contains(F.lower(F.lit(kw)))
        cond = expr if cond is None else (cond | expr)

    targets = (sheet_list_df
        .where(cond)
        .select("sheet_txt")
        .orderBy("sheet_txt")
        .collect()
    )
    return [r["sheet_txt"] for r in targets]
"""

In [0]:
"""
all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # mes (en, fe, ma, etc.)
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    # 1) leo índice y saco hojas candidatas
    sheet_list_df = get_sheet_candidates_from_indice(file_path)
    target_sheets = pick_target_sheets(sheet_list_df)

    if len(target_sheets) == 0:
        raise Exception(f"No encontré hojas objetivo en {file_name}. Revisa TARGET_KEYWORDS.")

    # 2) leo cada hoja objetivo y la apilo
    for sh in target_sheets:
        df = read_excel_sheet(file_path, sh)

        df = (df.withColumn("source_file", lit(file_name))
                .withColumn("periodo_mes", lit(periodo))
                .withColumn("sheet_name", lit(sh)))

        all_dfs.append(df)

if len(all_dfs) == 0:
    raise Exception("No se pudo leer ningún excel/hoja objetivo.")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(30))
print("Total filas:", df_final.count())
"""

In [0]:
"""
test_path = files[0]
df_test = read_indice_sheet(test_path)

print("Columnas:", df_test.columns)
display(df_test.limit(40))
"""

In [0]:
"""
# celda5
all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # Extrae mes: en2024, fe2024, etc (en tu nombre es -en2024, -fe2024...)
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    #df = read_excel_any_sheet(file_path)
    df = read_excel_sheet(file_path, "3")

    df = (df.withColumn("source_file", lit(file_name))
            .withColumn("periodo_mes", lit(periodo)))

    all_dfs.append(df)

if len(all_dfs) == 0:
    raise Exception("La lista all_dfs quedó vacía (no se pudo leer ningún excel).")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(20))
print("Total filas (aprox):", df_final.count())
"""

In [0]:
"""
test_path = files[0]
df_test = read_excel_sheet(test_path, "Ratios de Morosidad según días de incumplimiento")
display(df_test.limit(20))
print(df_test.columns)
"""

In [0]:
""" DENTRO DE LAS PRUEABAS, FUNCIONA PERO AÚN FALTA PULIR CELDA5
import re
import pandas as pd
import os
import tempfile
from pyspark.sql import functions as F
from pyspark.sql.functions import lit

def read_excel_sheet(path: str, sheet: str):
    ext = path.split(".")[-1].lower()
    
    # Para archivos .XLS (formato antiguo), usar pandas con xlrd
    if ext == "xls":
        # Crear archivo temporal
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".xls")
        temp_path = temp_file.name
        temp_file.close()
        
        try:
            # Copiar archivo de ABFSS a local temporal
            dbutils.fs.cp(path, f"file://{temp_path}")
            
            # Primero, leer el Índice para obtener el mapeo de nombres a números
            df_indice = pd.read_excel(temp_path, sheet_name="Índice", header=None, engine='xlrd')
            
            # Buscar el número de hoja correspondiente al nombre solicitado
            # El índice tiene el nombre en columna 1 y el número en columna 9
            sheet_number = None
            for idx, row in df_indice.iterrows():
                if pd.notna(row[1]) and pd.notna(row[9]):
                    sheet_name_in_index = str(row[1]).strip()
                    if sheet.lower() in sheet_name_in_index.lower() or sheet_name_in_index.lower() in sheet.lower():
                        sheet_number = int(row[9])
                        break
            
            if sheet_number is None:
                raise Exception(
                    f"No se encontró la hoja '{sheet}' en el índice del archivo {path}\n"
                    f"Revisa el nombre exacto en la hoja 'Índice' del Excel"
                )
            
            # Leer la hoja usando el número encontrado
            pdf = pd.read_excel(
                temp_path,
                sheet_name=str(sheet_number),
                header=None,
                engine='xlrd'
            )
            
            # Convertir a Spark DataFrame
            df = spark.createDataFrame(pdf)
            
            return df
            
        finally:
            # Limpiar archivo temporal
            if os.path.exists(temp_path):
                os.remove(temp_path)
    
    # Para archivos .XLSX, usar spark-excel
    reader = (spark.read.format("com.crealytics.spark.excel")
              .option("header", "false")
              .option("inferSchema", "false")
              .option("treatEmptyValuesAsNulls", "true")
              .option("maxColumns", "120")
              .option("maxRowsInMemory", "20000")
              .option("sheetName", sheet))
    
    return reader.load(path)
"""

In [0]:
""" DENTRO DE LAS PRUEABAS, FUNCIONA PERO AÚN FALTA PULIR CELDA6
# 1) Define las hojas objetivo (ajusta si en tu Excel el nombre exacto varía)
TARGET_SHEETS = [
    "Ratios de Morosidad según días de incumplimiento",
    "Morosidad por tipo de crédito y modalidad"
]

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    for sh in TARGET_SHEETS:
        df = read_excel_sheet(file_path, sh)

        df = (df.withColumn("source_file", lit(file_name))
                .withColumn("periodo_mes", lit(periodo))
                .withColumn("sheet_name", lit(sh)))

        all_dfs.append(df)

if len(all_dfs) == 0:
    raise Exception("No se pudo leer ningún excel/hoja objetivo.")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(30))
print("Total filas:", df_final.count())
"""

In [0]:
"""
import re
import pandas as pd
import os
import tempfile
from pyspark.sql.functions import lit

# Cache en memoria para no leer el índice 100 veces
_indice_cache = {}

def _download_to_tmp(abfss_path: str, suffix: str = ".xls") -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp_path = tmp.name
    tmp.close()
    dbutils.fs.cp(abfss_path, f"file:{tmp_path}")
    return tmp_path

def _build_sheet_map_from_indice(local_xls_path: str) -> dict:
    """
    Devuelve un dict: { 'texto_de_indice' : numero_hoja }
    El índice tiene el título en columna 1 y el número en columna 9.
    """
    # prueba nombres comunes de la hoja índice
    for indice_name in ["Índice", "INDICE", "ÍNDICE", "Indice", "indice", "índice"]:
        try:
            ind = pd.read_excel(local_xls_path, sheet_name=indice_name, header=None, engine="xlrd")
            break
        except Exception:
            ind = None
    if ind is None:
        raise Exception("No pude leer la hoja Índice con pandas/xlrd (nombre no encontrado).")

    sheet_map = {}
    for _, row in ind.iterrows():
        # El título está en columna 1, el número en columna 9
        if pd.notna(row[1]) and pd.notna(row[9]):
            title = str(row[1]).strip()
            try:
                num = int(row[9])
                if 1 <= num <= 60 and len(title) > 3:
                    sheet_map[title] = num
            except (ValueError, TypeError):
                continue

    if len(sheet_map) == 0:
        raise Exception("Leí la hoja Índice pero no pude construir el mapeo (título -> número).")

    return sheet_map

def get_sheet_number_from_title(path_abfss: str, target_title: str) -> int:
    """
    Encuentra el número de hoja para un título objetivo usando el índice.
    Usa coincidencia 'contains' (case-insensitive).
    """
    if path_abfss in _indice_cache:
        sheet_map = _indice_cache[path_abfss]
    else:
        local_path = _download_to_tmp(path_abfss, suffix=".xls")
        try:
            sheet_map = _build_sheet_map_from_indice(local_path)
            _indice_cache[path_abfss] = sheet_map
        finally:
            if os.path.exists(local_path):
                os.remove(local_path)

    # match flexible
    tgt = target_title.lower()
    for title, num in sheet_map.items():
        if tgt in title.lower():
            return num

    # si no matchea, mostramos pistas
    suggestions = [t for t in sheet_map.keys() if any(w in t.lower() for w in tgt.split()[:2])]
    raise Exception(f"No encontré '{target_title}' en índice. Ejemplos cercanos: {suggestions[:8]}")

def read_xls_sheet_by_number(path_abfss: str, sheet_number: int):
    """
    Lee hoja .XLS por número usando pandas/xlrd y devuelve Spark DF.
    """
    local_path = _download_to_tmp(path_abfss, suffix=".xls")
    try:
        pdf = pd.read_excel(local_path, sheet_name=sheet_number, header=None, engine="xlrd")
        return spark.createDataFrame(pdf)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)
"""

In [0]:
"""
import pandas as pd, tempfile, os

one = files[0]
tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".xls").name
dbutils.fs.cp(one, f"file:{tmp}")

# prueba nombres de índice
indice_name = None
for n in ["Índice", "INDICE", "ÍNDICE", "Indice", "indice", "índice"]:
    try:
        ind = pd.read_excel(tmp, sheet_name=n, header=None, engine="xlrd")
        indice_name = n
        break
    except:
        pass

print("Índice sheet:", indice_name)
print("shape:", ind.shape)
print("primeras 5 filas, primeras 15 columnas:")
display(ind.iloc[:5, :15])

# mira si hay números 1..60 en alguna columna
nums = {}
for c in range(min(25, ind.shape[1])):
    col = ind.iloc[:, c]
    # cuenta cuántos valores parecen números 1..60
    cnt = 0
    for v in col.dropna().head(200).tolist():
        s = str(v).strip()
        if s.isdigit():
            k = int(s)
            if 1 <= k <= 60:
                cnt += 1
    nums[c] = cnt

print("conteo de posibles números por columna (top):", sorted(nums.items(), key=lambda x: -x[1])[:10])

os.remove(tmp)
"""

# INICIANDO

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("storageName", "adlsbsriskdev")
dbutils.widgets.text("catalogName", "sbsrisk_dev")
dbutils.widgets.text("year", "2024")   # cambiar el año de procesamiento

storage = dbutils.widgets.get("storageName")
catalog = dbutils.widgets.get("catalogName")
year = dbutils.widgets.get("year")

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")

In [0]:
raw_base = f"abfss://raw@{storage}.dfs.core.windows.net/sbs/{year}/"
display(dbutils.fs.ls(raw_base))

In [0]:
files = [f.path for f in dbutils.fs.ls(raw_base) 
         if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]

print("Total excels encontrados:", len(files))
for f in files[:5]:
    print(f)

if len(files) == 0:
    raise Exception(f"No se encontraron archivos .XLS/.XLSX en: {raw_base}")

In [0]:
# Dependencias para leer xls
%pip install xlrd==2.0.1

In [0]:
import re
import pandas as pd
import os
import tempfile
from pyspark.sql.functions import lit

_indice_cache = {}

def _download_to_tmp(abfss_path: str, suffix: str = ".xls") -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp_path = tmp.name
    tmp.close()
    dbutils.fs.cp(abfss_path, f"file:{tmp_path}")
    return tmp_path

def _build_sheet_map_from_indice(local_xls_path: str) -> dict:
    ind = None
    for indice_name in ["Índice", "INDICE", "ÍNDICE", "Indice", "indice", "índice"]:
        try:
            ind = pd.read_excel(local_xls_path, sheet_name=indice_name, header=None, engine="xlrd")
            break
        except:
            pass
    if ind is None:
        raise Exception("No pude leer la hoja Índice con pandas/xlrd (nombre no encontrado).")

    sheet_map = {}

    for _, row in ind.iterrows():
        values = row.tolist()

        num = None
        for v in values:
            if pd.isna(v): 
                continue
            try:
                k = int(float(v))
                if 1 <= k <= 60:
                    num = k
                    break
            except (ValueError, TypeError):
                continue

        if num is None:
            continue

        text_candidates = []
        for v in values:
            if pd.isna(v):
                continue
            sv = str(v).strip()
            if len(sv) >= 6:
                try:
                    float(sv)
                except ValueError:
                    text_candidates.append(sv)

        if not text_candidates:
            continue

        title = max(text_candidates, key=len)
        sheet_map[title] = num

    if len(sheet_map) == 0:
        raise Exception("Leí la hoja Índice pero no pude construir el mapeo (título -> número).")

    return sheet_map

def get_sheet_number_from_title(path_abfss: str, target_title: str) -> int:
    if path_abfss in _indice_cache:
        sheet_map = _indice_cache[path_abfss]
    else:
        local_path = _download_to_tmp(path_abfss, suffix=".xls")
        try:
            sheet_map = _build_sheet_map_from_indice(local_path)
            _indice_cache[path_abfss] = sheet_map
        finally:
            if os.path.exists(local_path):
                os.remove(local_path)

    tgt = target_title.lower()
    for title, num in sheet_map.items():
        if tgt in title.lower():
            return num

    suggestions = [t for t in sheet_map.keys() if any(w in t.lower() for w in tgt.split()[:2])]
    raise Exception(f"No encontré '{target_title}' en índice. Ejemplos cercanos: {suggestions[:8]}")

def read_xls_sheet_by_number(path_abfss: str, sheet_number: int):
    local_path = _download_to_tmp(path_abfss, suffix=".xls")
    try:
        pdf = pd.read_excel(local_path, sheet_name=sheet_number, header=None, engine="xlrd")
        pdf = pdf.astype(str)
        return spark.createDataFrame(pdf)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

In [0]:
# Hojas objetivo por “título” tal como aparece en el índice (usa contains)
TARGET_TITLES = [
    "Ratios de Morosidad según días de incumplimiento",
    "Morosidad por tipo de crédito y modalidad"
]

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    for title in TARGET_TITLES:
        try:
            sheet_num = get_sheet_number_from_title(file_path, title)
            df = read_xls_sheet_by_number(file_path, sheet_num)

            df = (df.withColumn("source_file", lit(file_name))
                    .withColumn("periodo_mes", lit(periodo))
                    .withColumn("sheet_title", lit(title))
                    .withColumn("sheet_num", lit(int(sheet_num))))

            all_dfs.append(df)

        except Exception as e:
            # En algunos meses puede variar el título o faltar la tabla.
            print(f"[WARN] {file_name} - no pude leer '{title}': {e}")

if len(all_dfs) == 0:
    raise Exception("No se pudo leer ninguna hoja objetivo en ningún archivo.")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(30))
print("Total filas:", df_final.count())

In [0]:
from pyspark.sql import functions as F

# Quita filas donde TODAS las columnas de datos sean "nan" / "None" / vacío
data_cols = [c for c in df_final.columns if not c in ["source_file","periodo_mes","sheet_title","sheet_num"]]

df_bronze = df_final
for c in data_cols:
    df_bronze = df_bronze.withColumn(c, F.when(F.col(c).isin("nan","None",""), None).otherwise(F.col(c)))

df_bronze = df_bronze.na.drop("all", subset=data_cols)

display(df_bronze.limit(30))
print("Filas luego de limpiar vacíos:", df_bronze.count())

In [0]:
bronze_path = f"abfss://bronze@{storage}.dfs.core.windows.net/{catalog}/bronze/bronze_sbs_morosidad_{year}"

(df_final.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(bronze_path))

print("OK escrito en:", bronze_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.bronze_sbs_morosidad_{year}
USING DELTA
LOCATION '{bronze_path}'
""")

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {catalog}.bronze.bronze_sbs_morosidad_{year}").show()

In [0]:
display(dbutils.fs.ls(bronze_path))